# Class 3. Measurement, Observables, and the Quantum Toolbox

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

Goals of this practice session:

1. Compute and interpret expectation values for single- and two-qubit states.
2. Understand shot-based estimation and confidence bounds.
3. Use Qiskit primitives (`StatevectorSampler`, `StatevectorEstimator`) correctly.
4. Connect quantum measurement features to a real-data classification pipeline.

**Convention reminder.** Unless stated otherwise, each qubit starts in $\lvert 0\rangle$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram

import pennylane as qml

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

---
## Background: observables, eigenpairs, Born rule

An **observable** is a Hermitian operator $O$.

- It has eigenpairs $(\lambda_i,\lvert\phi_i\rangle)$ and a spectral decomposition
  $O = \sum_i \lambda_i\lvert\phi_i\rangle\langle\phi_i\rvert$.
- A projective measurement of $O$ returns an eigenvalue $\lambda_i$ and leaves the system in the corresponding eigenstate $\lvert\phi_i\rangle$.

If the system is in $\lvert\psi\rangle$, the **Born rule** gives
\[
P(\lambda_i)=\lvert\langle\phi_i\vert\psi\rangle\rvert^2.
\]
For real unit vectors, $\lvert\langle\phi_i\vert\psi\rangle\rvert=\cos\theta$, hence $P=\cos^2\theta$.

Convention: $\langle\psi\vert=(\lvert\psi\rangle)^\dagger$ (conjugate transpose). In NumPy, `np.vdot(a, b)` computes $a^\dagger b$.

The **expectation value** is the mean outcome:
\[
\langle O\rangle = \langle\psi\vert O\vert\psi\rangle = \sum_i \lambda_i P(\lambda_i).
\]

### M3.0. Born rule as overlap; expectation as weighted average

For
$\lvert\psi\rangle = \cos(\pi/8)\lvert0\rangle + \sin(\pi/8)\lvert1\rangle$,
compute $P(0)$ and $P(1)$ via overlaps with $\lvert0\rangle,\lvert1\rangle$. Then compute $\langle Z\rangle$ as a weighted average of eigenvalues $\{+1,-1\}$ and verify it matches $\langle\psi\vert Z\vert\psi\rangle$.

In [ ]:
theta = np.pi / 8
psi = np.array([np.cos(theta), np.sin(theta)], dtype=complex)

ket0 = np.array([1.0, 0.0], dtype=complex)
ket1 = np.array([0.0, 1.0], dtype=complex)

p0 = np.abs(np.vdot(ket0, psi)) ** 2
p1 = np.abs(np.vdot(ket1, psi)) ** 2

Z = np.array([[1, 0], [0, -1]], dtype=complex)

ex_z_from_probs = (+1) * p0 + (-1) * p1
ex_z_direct = np.vdot(psi, Z @ psi)

print(f"P(0) = {p0:.6f}")
print(f"P(1) = {p1:.6f}")
print(f"P(0)+P(1) = {p0+p1:.6f}")
print(f"<Z> from probs:   {ex_z_from_probs:.6f}")
print(f"<Z> as <psi|Z|psi>: {np.real(ex_z_direct):.6f}")

---
## Part 1: Math tasks

### M3.1. Single-qubit expectation values

For
$\lvert\psi\rangle = \cos(\pi/8)\lvert0\rangle + \sin(\pi/8)\lvert1\rangle$,
compute $\langle Z\rangle$, $\langle X\rangle$, and $\langle Y\rangle$.

In [ ]:
theta = np.pi / 8
psi = np.array([np.cos(theta), np.sin(theta)], dtype=complex)

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

ex_z = np.vdot(psi, Z @ psi)
ex_x = np.vdot(psi, X @ psi)
ex_y = np.vdot(psi, Y @ psi)

print(f"<Z> = {np.real(ex_z):.6f}")
print(f"<X> = {np.real(ex_x):.6f}")
print(f"<Y> = {np.real(ex_y):.6f}")

### M3.2. Bell-state observables

For $\lvert\Phi^+\rangle = (\lvert00\rangle + \lvert11\rangle)/\sqrt{2}$,
compute $\langle Z\otimes Z\rangle$, $\langle Z\otimes I\rangle$, and $\langle X\otimes X\rangle$.

In [ ]:
phi_plus = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)

I = np.eye(2, dtype=complex)
ZZ = np.kron(Z, Z)
ZI = np.kron(Z, I)
XX = np.kron(X, X)

ex_zz = np.vdot(phi_plus, ZZ @ phi_plus)
ex_zi = np.vdot(phi_plus, ZI @ phi_plus)
ex_xx = np.vdot(phi_plus, XX @ phi_plus)

print(f"<ZZ> = {np.real(ex_zz):.6f}")
print(f"<ZI> = {np.real(ex_zi):.6f}")
print(f"<XX> = {np.real(ex_xx):.6f}")

### M3.3. Hoeffding bound and shot count

Use Hoeffding:
$N \ge \ln(2/\delta)/(2\epsilon^2)$,
for $\epsilon=0.01$, confidence $95\%$ ($\delta=0.05$).

In [ ]:
epsilon = 0.01
delta = 0.05
n_min = np.log(2 / delta) / (2 * epsilon**2)
print(f"Minimum N from Hoeffding bound: {int(np.ceil(n_min))}")

---
## Part 2: Programming tasks

### P3.1. Sampler vs Estimator on one circuit

Use `StatevectorSampler` for counts and `StatevectorEstimator` for expectation values.

In [ ]:
theta = 1.2

sampler = StatevectorSampler()
estimator = StatevectorEstimator()

qc_counts = QuantumCircuit(1)
qc_counts.ry(theta, 0)
qc_counts.measure_all()
counts = sampler.run([qc_counts], shots=1024).result()[0].data.meas.get_counts()
print("Sampler counts:", counts)

qc_ev = QuantumCircuit(1)
qc_ev.ry(theta, 0)
ev_z = estimator.run([(qc_ev, SparsePauliOp("Z"))]).result()[0].data.evs
print(f"Estimator <Z> = {float(np.real(ev_z)):.6f}")

In [ ]:
plot_histogram(counts)

### P3.2. PennyLane expectation value curve

Create a one-qubit $R_y(\theta)$ circuit and plot $\langle Z\rangle$.

In [ ]:
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def expval_z(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

thetas = np.linspace(0, 2 * np.pi, 120)
vals = np.array([expval_z(t) for t in thetas], dtype=float)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thetas, vals, label=r"$\langle Z\rangle$ (circuit)", lw=2)
ax.plot(thetas, np.cos(thetas), "--", label=r"$\cos\theta$", lw=1.8)
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$\langle Z\rangle$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### P3.3. Shot-scaling experiment

Estimate $\langle Z\rangle$ from counts for shots in `[10, 100, 1000, 10000]` and compare with exact.

In [ ]:
theta = np.pi / 4
true_ev = np.cos(theta)

qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()

shots_values = [10, 100, 1000, 10000]
estimates = []

for shots in shots_values:
    c = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    p0 = c.get("0", 0) / shots
    p1 = c.get("1", 0) / shots
    ev_hat = p0 - p1
    estimates.append(ev_hat)
    print(f"shots={shots:5d}  <Z>_hat={ev_hat:+.4f}  counts={c}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(shots_values, estimates, "o-", lw=2)
ax.axhline(true_ev, color="red", ls="--", label=f"exact={true_ev:+.4f}")
ax.set_xscale("log", base=10)
ax.set_xlabel("shots")
ax.set_ylabel(r"estimated $\langle Z\rangle$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### P3.4. Bell correlation check in Qiskit

Verify strong correlation on $\langle Z_0 Z_1\rangle$ for a Bell state.

In [ ]:
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

ev_zz = estimator.run([(qc_bell, SparsePauliOp("ZZ"))]).result()[0].data.evs
ev_zi = estimator.run([(qc_bell, SparsePauliOp("ZI"))]).result()[0].data.evs

print(f"<ZZ> = {float(np.real(ev_zz)):+.6f}")
print(f"<ZI> = {float(np.real(ev_zi)):+.6f}")

### P3.5. Real dataset mini task: Iris with quantum measurement features

Use two Iris classes (versicolor/virginica) and compare:

1. logistic regression on standardized raw features
2. logistic regression on quantum-derived features
$\left(\langle X\rangle,\langle Y\rangle,\langle Z\rangle\right)$ from one-qubit encoding.

In [ ]:
iris = load_iris()
X_all = iris.data[:, 2:4]
y_all = iris.target

mask = y_all != 0
X = X_all[mask]
y = (y_all[mask] == 2).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

def to_angles(X_in, X_ref):
    xmin = X_ref.min(axis=0)
    xmax = X_ref.max(axis=0)
    denom = np.where(np.abs(xmax - xmin) < 1e-12, 1.0, xmax - xmin)
    x01 = np.clip((X_in - xmin) / denom, 0.0, 1.0)
    return np.pi * x01

A_train = to_angles(X_train_s, X_train_s)
A_test = to_angles(X_test_s, X_train_s)

dev_feat = qml.device("default.qubit", wires=1)


@qml.qnode(dev_feat)
def q_features(a0, a1):
    qml.RY(a0, wires=0)
    qml.RZ(a1, wires=0)
    return (
        qml.expval(qml.PauliX(0)),
        qml.expval(qml.PauliY(0)),
        qml.expval(qml.PauliZ(0)),
    )

Q_train = np.array([q_features(a0, a1) for a0, a1 in A_train], dtype=float)
Q_test = np.array([q_features(a0, a1) for a0, a1 in A_test], dtype=float)

clf_raw = LogisticRegression(max_iter=2000)
clf_raw.fit(X_train_s, y_train)
acc_raw = accuracy_score(y_test, clf_raw.predict(X_test_s))

clf_q = LogisticRegression(max_iter=2000)
clf_q.fit(Q_train, y_train)
acc_q = accuracy_score(y_test, clf_q.predict(Q_test))

print(f"Raw standardized features accuracy: {acc_raw:.3f}")
print(f"Quantum-derived feature accuracy:   {acc_q:.3f}")

---
## Summary

Shot-based measurement estimates probabilities and expectation values with statistical uncertainty that decreases as $1/\sqrt{N}$. Observables specify what quantity is extracted from a state, and Qiskit's sampler/estimator primitives separate distribution questions from expectation-value questions.

The Iris mini task demonstrates an early real-data bridge: quantum-style measurement features can be used directly in a classical classifier.

---
## Optional extension: fidelity / SWAP test (preview)

For two pure states, fidelity is
\[
F(\psi,\varphi)=\lvert\langle\psi\vert\varphi\rangle\rvert^2.
\]
A SWAP-test style estimate uses the ancilla relation
\[
P(\mathrm{anc}=0)=\frac{1+F}{2}.
\]

Use this as a conceptual preview of kernel-style similarity measures.

In [ ]:
# Optional preview scaffold:
# implement a small SWAP-test style routine and compare an estimated
# fidelity with an exact statevector overlap.


---
## Optional extension: mixed states and depolarizing noise

For a mixed state (density matrix) $\rho$:

- $P(0)=\rho_{00}$ and $P(1)=\rho_{11}$ for a $Z$-basis measurement.
- Expectation values are computed as $\langle O\rangle=\mathrm{Tr}(\rho O)$.

A simple one-qubit depolarizing channel is
$\rho'=(1-p)\rho + p I/2$.
Show numerically that for Pauli $Z$, $\langle Z\rangle$ shrinks by a factor $(1-p)$.

In [ ]:
# Optional preview scaffold:
# build rho = |psi><psi|, apply a depolarizing channel,
# and verify how <Z> changes as p increases.
